In [1]:
%pip install requests pandas

  Obtaining dependency information for requests from https://files.pythonhosted.org/packages/a0/f4/c67b0b3f1b9245e8d266f0f112c500d50e5b4e83cb6f3b71b6528104182a/requests-2.34.2-py3-none-any.whl.metadata
  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/d3/dc/d2df02854aec5d47659acfb2be352eecc691845b2f86e99c84f1010a8671/pandas-3.0.6-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for charset_normalizer<4,>=2 from https://files.pythonhosted.org/packages/e8/fc/fdf8cf52ff21cd5bf158f20978991cf985325842f74283eb6df26c8a39d8/charset_normalizer-3.5.2-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/47.3 kB ? eta -:--:--
     ---------------------------------- ----- 41.0/47.3 kB 1.9 MB/s eta 0:00:01
     ---------------------------------------- 47.3/47.3 kB 1.2 MB/s eta 0:00:00
  Obtaining dependency information for idna<4,>=2.5 from https://files.pythonhosted.org/packages/58/a2/bb081bab032533a855d44


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
import pandas as pd

# Fetch the raw FPL bootstrap data
url = 'https://fantasy.premierleague.com/api/bootstrap-static/'
response = requests.get(url)
data = response.json()

# Extract player elements into a pandas DataFrame
df = pd.DataFrame(data['elements'])

# Display the first 5 rows and the total dimensions
print(f"Dataset shape: {df.shape[0]} players, {df.shape[1]} metrics per player")
df[['web_name', 'team', 'element_type', 'now_cost', 'total_points', 'goals_scored', 'assists', 'minutes']].head()

Dataset shape: 667 players, 109 metrics per player


,web_name,team,element_type,now_cost,total_points,goals_scored,assists,minutes
0,Raya,1,1,61,33,0,0,540
1,Arrizabalaga,1,1,49,0,0,0,0
2,Meslier,1,1,49,0,0,0,0
3,Gabriel,1,2,79,27,0,0,540
4,J.Timber,1,2,65,2,0,0,131


In [2]:
# 1. Sort the players by 'total_points' in descending order (highest to lowest)
df_sorted = df.sort_values(by='total_points', ascending=False)

# 2. Pick the columns we actually care about right now
columns_to_view = ['web_name', 'team', 'element_type', 'now_cost', 'total_points', 'goals_scored', 'assists', 'minutes']

# 3. Show the top 10 players
df_sorted[columns_to_view].head(10)

,web_name,team,element_type,now_cost,total_points,goals_scored,assists,minutes
136,Groß,5,3,60,53,3,5,540
127,De Cuyper,5,2,51,53,2,3,490
272,Tarkowski,9,2,62,46,1,1,540
491,Haaland,15,4,156,45,6,0,529
184,João Pedro,6,4,78,45,5,3,431
482,Cherki,15,3,79,44,4,2,392
403,Bogle,13,2,47,43,2,0,450
474,Gvardiol,15,2,58,41,1,4,458
151,Vuskovic,5,2,51,40,1,0,527
105,Schade,4,3,62,40,3,2,498


In [3]:
# 1. Grab the "teams" and "element_types" (positions) data we ignored earlier
teams_data = data['teams']
positions_data = data['element_types']

# 2. Convert them into their own mini-spreadsheets
teams_df = pd.DataFrame(teams_data)
positions_df = pd.DataFrame(positions_data)

# 3. Create mapping dictionaries (Think of these as translation guides)
# This maps Team ID (e.g., 1) to Team Name (e.g., 'Arsenal')
team_mapping = dict(zip(teams_df['id'], teams_df['name']))

# This maps Position ID (e.g., 3) to Position Name (e.g., 'Midfielder')
position_mapping = dict(zip(positions_df['id'], positions_df['singular_name']))

# 4. Apply the translation to our main player spreadsheet
df['team_name'] = df['team'].map(team_mapping)
df['position'] = df['element_type'].map(position_mapping)

# 5. Let's look at the cleaned-up top 10 players!
columns_to_view = ['web_name', 'team_name', 'position', 'now_cost', 'total_points', 'goals_scored', 'assists', 'minutes']
df_sorted = df.sort_values(by='total_points', ascending=False)
df_sorted[columns_to_view].head(10)

,web_name,team_name,position,now_cost,total_points,goals_scored,assists,minutes
136,Groß,Brighton,Midfielder,60,53,3,5,540
127,De Cuyper,Brighton,Defender,51,53,2,3,490
272,Tarkowski,Everton,Defender,62,46,1,1,540
491,Haaland,Man City,Forward,156,45,6,0,529
184,João Pedro,Chelsea,Forward,78,45,5,3,431
482,Cherki,Man City,Midfielder,79,44,4,2,392
403,Bogle,Leeds,Defender,47,43,2,0,450
474,Gvardiol,Man City,Defender,58,41,1,4,458
151,Vuskovic,Brighton,Defender,51,40,1,0,527
105,Schade,Brentford,Midfielder,62,40,3,2,498


In [4]:
# 1. Get the ID and name of the #1 overall player from our sorted list
top_player_id = df_sorted.iloc[0]['id']
top_player_name = df_sorted.iloc[0]['web_name']
print(f"Fetching match history for: {top_player_name} (ID: {top_player_id})")

# 2. Ping the specific FPL endpoint for individual player history
history_url = f'https://fantasy.premierleague.com/api/element-summary/{top_player_id}/'
response = requests.get(history_url)
player_history = response.json()['history']

# 3. Convert their match history into a DataFrame
history_df = pd.DataFrame(player_history)

# 4. The Magic Math: Calculate a 3-game rolling average of their points
# .rolling(window=3) tells pandas to group the current game with the 2 previous games
# .mean() calculates the average of those 3 games
history_df['3_game_avg_points'] = history_df['total_points'].rolling(window=3).mean()

# 5. View the most recent 5 games using .tail()
# 'round' is the FPL term for Gameweek
columns_to_view = ['round', 'minutes', 'goals_scored', 'total_points', '3_game_avg_points']
history_df[columns_to_view].tail(5)

Fetching match history for: Groß (ID: 124)


,round,minutes,goals_scored,total_points,3_game_avg_points
1,2,90,1,13,NaN
2,3,90,0,1,5.333333
3,4,90,1,17,10.333333
4,5,90,1,14,10.666667
5,6,90,0,6,12.333333


In [5]:
import time

# Create an empty list to hold every player's history dataframe
all_players_history = []

# Get a list of every player ID in the league
player_ids = df['id'].tolist()

print(f"Starting fetch for {len(player_ids)} players. This will take a couple of minutes...")

# Loop through every ID
for index, player_id in enumerate(player_ids):
    
    # 1. Fetch the data
    url = f'https://fantasy.premierleague.com/api/element-summary/{player_id}/'
    response = requests.get(url)
    
    # 2. Check if the request was successful
    if response.status_code == 200:
        player_history = response.json().get('history', [])
        
        # Only process players who actually have match history
        if player_history:
            history_df = pd.DataFrame(player_history)
            
            # Add their ID so we know who this data belongs to
            history_df['element'] = player_id 
            
            # Calculate the 3-game rolling average
            history_df['3_game_avg_points'] = history_df['total_points'].rolling(window=3).mean()
            
            # Add this player's dataframe to our master list
            all_players_history.append(history_df)
    
    # 3. Polite pausing to avoid API bans (0.2 seconds between requests)
    time.sleep(0.3)
    
    # Print a progress update every 50 players
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1}/{len(player_ids)} players...")

print("Fetch complete! Stacking the data...")

# 4. Combine all the individual mini-spreadsheets into one massive DataFrame
master_history_df = pd.concat(all_players_history, ignore_index=True)

# 5. Let's look at the size of our new master dataset
print(f"Master Dataset Shape: {master_history_df.shape[0]} total player-matches recorded.")

Starting fetch for 667 players. This will take a couple of minutes...
Processed 50/667 players...
Processed 100/667 players...
Processed 150/667 players...
Processed 200/667 players...
Processed 250/667 players...
Processed 300/667 players...
Processed 350/667 players...
Processed 400/667 players...
Processed 450/667 players...
Processed 500/667 players...
Processed 550/667 players...


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [6]:
import time
import requests

all_players_history = []
player_ids = df['id'].tolist()

print(f"Starting fetch for {len(player_ids)} players. Adding retry logic...")

for index, player_id in enumerate(player_ids):
    url = f'https://fantasy.premierleague.com/api/element-summary/{player_id}/'
    
    # --- NEW: Retry Logic ---
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # Added a 10-second timeout so the script doesn't hang forever
            response = requests.get(url, timeout=10) 
            
            if response.status_code == 200:
                player_history = response.json().get('history', [])
                
                if player_history:
                    history_df = pd.DataFrame(player_history)
                    history_df['element'] = player_id 
                    history_df['3_game_avg_points'] = history_df['total_points'].rolling(window=3).mean()
                    all_players_history.append(history_df)
            
            # If the request succeeds, break out of the retry loop and move to the next player
            break 
            
        except requests.exceptions.RequestException as e:
            # If the server drops the connection, this block catches the crash
            print(f"  [!] Connection error on player {player_id}. Retrying in 3 seconds... ({attempt + 1}/{max_retries})")
            time.sleep(3) 
    # ------------------------
    
    # Slightly longer baseline pause to keep the server happy
    time.sleep(0.3)
    
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1}/{len(player_ids)} players...")

print("Fetch complete! Stacking the data...")
master_history_df = pd.concat(all_players_history, ignore_index=True)
print(f"Master Dataset Shape: {master_history_df.shape[0]} total player-matches recorded.")

Starting fetch for 667 players. Adding retry logic...
Processed 50/667 players...
Processed 100/667 players...
Processed 150/667 players...
Processed 200/667 players...
Processed 250/667 players...
Processed 300/667 players...
Processed 350/667 players...
Processed 400/667 players...
Processed 450/667 players...
Processed 500/667 players...
Processed 550/667 players...
Processed 600/667 players...
Processed 650/667 players...
Fetch complete! Stacking the data...
Master Dataset Shape: 3883 total player-matches recorded.


In [7]:
# 1. Map the opponent's ID number to their actual team name
# We can reuse the 'team_mapping' dictionary we built in Phase 1
master_history_df['opponent_name'] = master_history_df['opponent_team'].map(team_mapping)

# 2. Convert the 'was_home' column from True/False to a 1 or 0
# Machine learning models calculate math much faster with 1s and 0s
master_history_df['home_game'] = master_history_df['was_home'].astype(int)

# 3. Bring in the player's name from our original dataset
# We map the 'element' (player ID) to their name
player_mapping = dict(zip(df['id'], df['web_name']))
master_history_df['player_name'] = master_history_df['element'].map(player_mapping)

# 4. Let's look at Erling Haaland's recent games as an example
# (If Haaland isn't in your top list, swap the name for whoever is #1)
columns_to_show = ['player_name', 'round', 'opponent_name', 'home_game', 'minutes', '3_game_avg_points', 'total_points']

# Filter the massive dataset for just one player to verify our math worked
sample_player_data = master_history_df[master_history_df['player_name'] == 'Haaland']
sample_player_data[columns_to_show].tail(5)

,player_name,round,opponent_name,home_game,minutes,3_game_avg_points,total_points
2862,Haaland,2,Crystal Palace,0,90,NaN,13
2863,Haaland,3,Coventry City,1,90,8.000000,9
2864,Haaland,4,Man Utd,0,90,10.333333,9
2865,Haaland,5,Sunderland,1,90,8.000000,6
2866,Haaland,6,Liverpool,0,79,7.000000,6


In [8]:
# 1. Sort the data chronologically. 
# This ensures Gameweek 1 is followed by Gameweek 2 for the correct player.
master_history_df = master_history_df.sort_values(by=['element', 'round'])

# 2. Create the target variable by shifting the points UP by one row (-1)
# We use .groupby('element') so we don't accidentally shift Player B's Gameweek 1 points into Player A's final game.
master_history_df['target_next_week_points'] = master_history_df.groupby('element')['total_points'].shift(-1)

# 3. View the shifted data for our sample player
columns_to_show = ['player_name', 'round', 'total_points', 'target_next_week_points']
sample_player_data = master_history_df[master_history_df['player_name'] == 'Haaland']

# We use .tail(5) to see the very end of their match history
sample_player_data[columns_to_show].tail(5)

,player_name,round,total_points,target_next_week_points
2862,Haaland,2,13,9.0
2863,Haaland,3,9,9.0
2864,Haaland,4,9,6.0
2865,Haaland,5,6,6.0
2866,Haaland,6,6,NaN


In [9]:
%pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/29/8a/8d1e888fe3d0dd4e2a1f3356dad703eed569aaf673e222b6f647005ad30f/scikit_learn-1.9.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/95/da/0d1df507cf574b3f224ccc3d45244c9a1d732c81dcb26b1e8a766ae271a8/scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 3.2 MB/s eta 0:00:00
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/18/53/84099323c2ec4be98d935f63c033ac4151ee83836ca1050ede3b3aadf155/joblib-1.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/d1/89/5d4c86da1130d9059681e5b6cd7645df5c10279a6a079c5c37dcb2cc6f3f/narwhals-2.27.1-py3-none-any.whl.metadata
  Ob


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Clean the Data for Training
# We must drop rows with NaN values. We can't train on future games (where target is NaN),
# and we can't train on Gameweek 1 (where the 3-game average is NaN).
trainable_df = master_history_df.dropna(subset=['target_next_week_points', '3_game_avg_points'])

# 2. Define the Features (X) and the Target (y)
# X = The clues the model uses to guess
features = ['3_game_avg_points', 'home_game', 'minutes']
X = trainable_df[features]

# y = The exact answer it is trying to predict
y = trainable_df['target_next_week_points']

# 3. Split the Data (80% for Training, 20% for Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} historical matches...")
print(f"Testing against {len(X_test)} hidden matches...")

# 4. Initialize the Model
# n_estimators=100 means we are building a forest with 100 decision trees
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 5. Train the Model!
rf_model.fit(X_train, y_train)

print("\nModel successfully trained! 🧠⚽")

Training on 1512 historical matches...
Testing against 378 hidden matches...

Model successfully trained! 🧠⚽


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Ask the model to predict the outcomes for the hidden 20% of data
predictions = rf_model.predict(X_test)

# 2. Grade the predictions against the actual points scored
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Mean Absolute Error (MAE): {mae:.2f} points")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} points")

Mean Absolute Error (MAE): 1.22 points
Root Mean Squared Error (RMSE): 2.36 points


In [12]:
# 1. Add Opponent Strength
# The FPL 'teams' data we pulled in Phase 1 already ranks every team's strength from 1 to 5.
# We create a dictionary to map the opponent's ID to their official strength rating.
opponent_strength_map = dict(zip(teams_df['id'], teams_df['strength']))
master_history_df['opponent_strength'] = master_history_df['opponent_team'].map(opponent_strength_map)

# 2. Filter out the noise
# We only want to train the model on games where the player actually stepped on the pitch.
filtered_df = master_history_df[master_history_df['minutes'] > 0].copy()

# 3. Clean the new dataset for training (Drop the NaNs)
trainable_df = filtered_df.dropna(subset=['target_next_week_points', '3_game_avg_points'])

# 4. Define our NEW upgraded features
features = ['3_game_avg_points', 'home_game', 'minutes', 'opponent_strength']
X = trainable_df[features]
y = trainable_df['target_next_week_points']

# 5. Split, Train, and Test Again
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train V2 of the model
rf_model_v2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_v2.fit(X_train, y_train)

# Grade V2
predictions_v2 = rf_model_v2.predict(X_test)
mae_v2 = mean_absolute_error(y_test, predictions_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_test, predictions_v2))

print(f"V2 Mean Absolute Error (MAE): {mae_v2:.2f} points")
print(f"V2 Root Mean Squared Error (RMSE): {rmse_v2:.2f} points")

V2 Mean Absolute Error (MAE): 2.11 points
V2 Root Mean Squared Error (RMSE): 3.02 points


In [13]:
# 1. Isolate the "Future" games
# We find the rows where our target is NaN (the upcoming gameweek)
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()

# 2. We still need to drop rows where the 3-game average is missing
# (e.g., players who just transferred to the league and haven't played 3 games yet)
future_games_df = future_games_df.dropna(subset=['3_game_avg_points'])

# 3. Isolate the features the model needs to make its guess
X_future = future_games_df[features]

# 4. Generate the predictions!
future_predictions = rf_model_v2.predict(X_future)

# 5. Add the predictions back into our spreadsheet so we can read them
future_games_df['projected_points'] = future_predictions

# 6. Sort the list to see who the AI thinks will score the most points next weekend
top_projections = future_games_df.sort_values(by='projected_points', ascending=False)

columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']
top_projections[columns_to_show].head(15)

,player_name,opponent_name,home_game,3_game_avg_points,projected_points
1223,Rudoni,Newcastle,1,4.666667,9.925409
798,Groß,Sunderland,0,12.333333,9.248095
197,Mings,Brentford,1,1.333333,7.979275
1893,Egan,Everton,1,1.333333,7.979275
2073,Mendy,Everton,1,1.333333,7.979275
2122,Walton,Fulham,1,1.333333,7.979275
3864,Isidor,Brighton,1,1.333333,7.979275
1005,Colwill,Bournemouth,1,1.333333,7.979275
2664,Munoz,Man City,1,1.333333,7.979275
870,Kostoulas,Sunderland,0,7.000000,7.868333


In [14]:
import numpy as np

# 1. Fetch the upcoming fixtures from the FPL API
fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?future=1'
fixtures_res = requests.get(fixtures_url).json()

# Isolate just the matches for the immediate next Gameweek
next_event = fixtures_res[0]['event'] 
next_fixtures = [f for f in fixtures_res if f['event'] == next_event]

# 2. Build a mapping guide for who plays who next week
next_match_map = {}
for match in next_fixtures:
    # If the team is playing at home
    next_match_map[match['team_h']] = {'opponent_id': match['team_a'], 'home_game': 1}
    # If the team is playing away
    next_match_map[match['team_a']] = {'opponent_id': match['team_h'], 'home_game': 0}

# 3. Bring in our future_games_df and attach the player's team ID
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()
player_team_map = dict(zip(df['id'], df['team']))
future_games_df['player_team_id'] = future_games_df['element'].map(player_team_map)

# 4. FIX THE WRONG WEEK BUG: Apply the future schedule to our dataset
future_games_df['future_opponent_id'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('opponent_id', np.nan))
future_games_df['home_game'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('home_game', np.nan))

# Map the new opponent's name and strength
future_games_df['opponent_name'] = future_games_df['future_opponent_id'].map(team_mapping)
future_games_df['opponent_strength'] = future_games_df['future_opponent_id'].map(opponent_strength_map)

# 5. FIX THE BENCH BUG: Filter out players who played 0 minutes in their last match
# We also drop players on a "Blank Gameweek" (no upcoming fixture)
future_games_df = future_games_df[future_games_df['minutes'] > 0]
future_games_df = future_games_df.dropna(subset=['opponent_strength', '3_game_avg_points'])

# 6. Generate the corrected predictions
X_future = future_games_df[features] # features = ['3_game_avg_points', 'home_game', 'minutes', 'opponent_strength']
future_games_df['projected_points'] = rf_model_v2.predict(X_future)

# 7. View the real Top 15 projections
top_projections = future_games_df.sort_values(by='projected_points', ascending=False)
columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']
top_projections[columns_to_show].head(15)

ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by RandomForestRegressor.

In [15]:
import numpy as np

# 1. Fetch the upcoming fixtures from the FPL API
fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?future=1'
fixtures_res = requests.get(fixtures_url).json()

# --- THE FIX: Safely find the next actual Gameweek ---
# Grab all event numbers, ignoring the 'None' unscheduled matches
valid_events = [f['event'] for f in fixtures_res if f.get('event') is not None]
next_event = min(valid_events) # The lowest number is the immediate next Gameweek

next_fixtures = [f for f in fixtures_res if f.get('event') == next_event]
# -----------------------------------------------------

# 2. Build a mapping guide for who plays who next week
next_match_map = {}
for match in next_fixtures:
    next_match_map[match['team_h']] = {'opponent_id': match['team_a'], 'home_game': 1}
    next_match_map[match['team_a']] = {'opponent_id': match['team_h'], 'home_game': 0}

# 3. Bring in our future_games_df and attach the player's team ID
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()
player_team_map = dict(zip(df['id'], df['team']))
future_games_df['player_team_id'] = future_games_df['element'].map(player_team_map)

# 4. Apply the future schedule to our dataset
future_games_df['future_opponent_id'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('opponent_id', np.nan))
future_games_df['home_game'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('home_game', np.nan))

# Map the new opponent's name and strength
future_games_df['opponent_name'] = future_games_df['future_opponent_id'].map(team_mapping)
future_games_df['opponent_strength'] = future_games_df['future_opponent_id'].map(opponent_strength_map)

# 5. Filter out players who played 0 minutes in their last match
# Also drop players on a "Blank Gameweek" (no upcoming fixture)
future_games_df = future_games_df[future_games_df['minutes'] > 0]
future_games_df = future_games_df.dropna(subset=['opponent_strength', '3_game_avg_points'])

# 6. Generate the corrected predictions
X_future = future_games_df[features] 
future_games_df['projected_points'] = rf_model_v2.predict(X_future)

# 7. View the real Top 15 projections
top_projections = future_games_df.sort_values(by='projected_points', ascending=False)
columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']
top_projections[columns_to_show].head(15)

ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by RandomForestRegressor.

In [16]:
import numpy as np

# 1. Start fresh with the future games
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()
print(f"1. Initial players looking for a next game: {len(future_games_df)}")

# 2. Get the fixtures
fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?future=1'
fixtures_res = requests.get(fixtures_url).json()

valid_events = [f['event'] for f in fixtures_res if f.get('event') is not None]
next_event = min(valid_events)
next_fixtures = [f for f in fixtures_res if f.get('event') == next_event]
print(f"2. Found {len(next_fixtures)} fixtures for Gameweek {next_event}")

# 3. Build mapping
next_match_map = {}
for match in next_fixtures:
    next_match_map[match['team_h']] = {'opponent_id': match['team_a'], 'home_game': 1}
    next_match_map[match['team_a']] = {'opponent_id': match['team_h'], 'home_game': 0}

# 4. Map Team IDs safely
player_team_map = dict(zip(df['id'], df['team']))
future_games_df['player_team_id'] = future_games_df['element'].map(player_team_map)

# 5. Map the Future Opponent
# We use .fillna(-1) to easily track who didn't get an opponent assigned
future_games_df['future_opponent_id'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('opponent_id', -1))
players_with_opponents = len(future_games_df[future_games_df['future_opponent_id'] != -1])
print(f"3. Players who successfully matched with an upcoming opponent: {players_with_opponents}")

# 6. Apply Strength and clean missing values
future_games_df['opponent_strength'] = future_games_df['future_opponent_id'].map(opponent_strength_map)
future_games_df = future_games_df.dropna(subset=['opponent_strength', '3_game_avg_points'])
print(f"4. Players remaining after dropping missing data: {len(future_games_df)}")

# 7. Filter minutes
future_games_df = future_games_df[future_games_df['minutes'] > 0]
print(f"5. Players remaining after dropping 0-minute benchwarmers: {len(future_games_df)}")

1. Initial players looking for a next game: 667
2. Found 10 fixtures for Gameweek 7
3. Players who successfully matched with an upcoming opponent: 667
4. Players remaining after dropping missing data: 0
5. Players remaining after dropping 0-minute benchwarmers: 0


In [17]:
import requests
import pandas as pd
import numpy as np

# 1. Fetch fresh mapping data to guarantee it exists in memory
bootstrap_data = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
teams_df = pd.DataFrame(bootstrap_data['teams'])
team_mapping = dict(zip(teams_df['id'], teams_df['name']))
opponent_strength_map = dict(zip(teams_df['id'], teams_df['strength']))
player_team_map = dict(zip(pd.DataFrame(bootstrap_data['elements'])['id'], pd.DataFrame(bootstrap_data['elements'])['team']))

# 2. Get upcoming fixtures
fixtures_res = requests.get('https://fantasy.premierleague.com/api/fixtures/?future=1').json()
valid_events = [f['event'] for f in fixtures_res if f.get('event') is not None]
next_event = min(valid_events)
next_fixtures = [f for f in fixtures_res if f.get('event') == next_event]

# 3. Build fixture map
next_match_map = {}
for match in next_fixtures:
    next_match_map[match['team_h']] = {'opponent_id': match['team_a'], 'home_game': 1}
    next_match_map[match['team_a']] = {'opponent_id': match['team_h'], 'home_game': 0}

# 4. Prep future dataset
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()
future_games_df['player_team_id'] = future_games_df['element'].map(player_team_map)

# 5. Map future opponent securely 
future_games_df['future_opponent_id'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('opponent_id', -1))

# Drop teams on a "Blank Gameweek" (no upcoming fixture)
future_games_df = future_games_df[future_games_df['future_opponent_id'] != -1]

# Apply mappings
future_games_df['opponent_strength'] = future_games_df['future_opponent_id'].map(opponent_strength_map)
future_games_df['home_game'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('home_game', 0))
future_games_df['opponent_name'] = future_games_df['future_opponent_id'].map(team_mapping)

# 6. Filter active players
future_games_df = future_games_df[future_games_df['minutes'] > 0]
future_games_df = future_games_df.dropna(subset=['opponent_strength', '3_game_avg_points'])

# 7. Predict using V2 Model
X_future = future_games_df[['3_game_avg_points', 'home_game', 'minutes', 'opponent_strength']]
future_games_df['projected_points'] = rf_model_v2.predict(X_future)

# Display the Top 15 targets
top_projections = future_games_df.sort_values(by='projected_points', ascending=False)
columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']
top_projections[columns_to_show].head(15)


ValueError: Found array with 0 sample(s) (shape=(0, 4)) while a minimum of 1 is required by RandomForestRegressor.

In [18]:
import requests
import pandas as pd
import numpy as np

# 1. Fetch fresh mapping data
bootstrap_data = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
teams_df = pd.DataFrame(bootstrap_data['teams'])

# Force ID columns to be integers to prevent dictionary mismatching
teams_df['id'] = teams_df['id'].astype(int)

team_mapping = dict(zip(teams_df['id'], teams_df['name']))
opponent_strength_map = dict(zip(teams_df['id'], teams_df['strength']))
player_team_map = dict(zip(pd.DataFrame(bootstrap_data['elements'])['id'].astype(int), pd.DataFrame(bootstrap_data['elements'])['team'].astype(int)))

# 2. Get upcoming fixtures
fixtures_res = requests.get('https://fantasy.premierleague.com/api/fixtures/?future=1').json()
valid_events = [f['event'] for f in fixtures_res if f.get('event') is not None]
next_event = min(valid_events)
next_fixtures = [f for f in fixtures_res if f.get('event') == next_event]

# 3. Build fixture map (Forcing integer keys)
next_match_map = {}
for match in next_fixtures:
    next_match_map[int(match['team_h'])] = {'opponent_id': int(match['team_a']), 'home_game': 1}
    next_match_map[int(match['team_a'])] = {'opponent_id': int(match['team_h']), 'home_game': 0}

# 4. Prep future dataset
future_games_df = master_history_df[master_history_df['target_next_week_points'].isna()].copy()
# Safely map and convert to int (filling missing teams with -1)
future_games_df['player_team_id'] = future_games_df['element'].map(player_team_map).fillna(-1).astype(int)

# 5. Map future opponent securely 
future_games_df['future_opponent_id'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('opponent_id', -1))

# Drop teams on a "Blank Gameweek" (no upcoming fixture)
future_games_df = future_games_df[future_games_df['future_opponent_id'] != -1]

# Apply mappings WITH fallbacks so the model never crashes on NaNs
future_games_df['opponent_strength'] = future_games_df['future_opponent_id'].map(opponent_strength_map).fillna(3) # Default to average strength 3
future_games_df['home_game'] = future_games_df['player_team_id'].apply(lambda x: next_match_map.get(x, {}).get('home_game', 0))
future_games_df['opponent_name'] = future_games_df['future_opponent_id'].map(team_mapping).fillna('Unknown')

# Fill missing form with 0 (for players who have played fewer than 3 total games)
future_games_df['3_game_avg_points'] = future_games_df['3_game_avg_points'].fillna(0)

# 6. Filter active players
future_games_df = future_games_df[future_games_df['minutes'] > 0]

# 7. Predict using V2 Model
X_future = future_games_df[['3_game_avg_points', 'home_game', 'minutes', 'opponent_strength']]
future_games_df['projected_points'] = rf_model_v2.predict(X_future)

# Display the Top 15 targets
top_projections = future_games_df.sort_values(by='projected_points', ascending=False)
columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']
top_projections[columns_to_show].head(15)


,player_name,opponent_name,home_game,3_game_avg_points,projected_points
1223,Rudoni,Spurs,0,4.666667,8.132738
3261,Horníček,Aston Villa,1,1.333333,7.979275
1486,Strand Larsen,Brighton,0,5.000000,7.729017
798,Groß,Crystal Palace,1,12.333333,7.663242
870,Kostoulas,Crystal Palace,1,7.000000,6.995000
2710,Barcola,Brentford,0,2.666667,6.942751
1546,Timber,Brighton,0,4.666667,6.525952
2146,O'Shea,Man City,0,4.666667,6.525952
77,Rice,Nott'm Forest,0,4.666667,6.525952
1145,Rushworth,Spurs,0,4.666667,6.525952


In [19]:
# 1. Define our V3 features (dropping 'minutes')
features_v3 = ['3_game_avg_points', 'home_game', 'opponent_strength']
X_v3 = trainable_df[features_v3]
y_v3 = trainable_df['target_next_week_points']

# 2. Split and Train Model V3
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(X_v3, y_v3, test_size=0.2, random_state=42)

rf_model_v3 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_v3.fit(X_train_v3, y_train_v3)

print("Model V3 Trained (Minutes feature removed).")

# 3. Generate Predictions for the upcoming Gameweek
X_future_v3 = future_games_df[features_v3] 
future_games_df['projected_points'] = rf_model_v3.predict(X_future_v3)

# 4. View the corrected Top 15 projections
top_projections_v3 = future_games_df.sort_values(by='projected_points', ascending=False)
columns_to_show = ['player_name', 'opponent_name', 'home_game', '3_game_avg_points', 'projected_points']

top_projections_v3[columns_to_show].head(15)

Model V3 Trained (Minutes feature removed).


,player_name,opponent_name,home_game,3_game_avg_points,projected_points
744,De Cuyper,Crystal Palace,1,10.666667,7.663242
798,Groß,Crystal Palace,1,12.333333,7.663242
251,McGinn,Newcastle,0,7.333333,6.686341
3027,Cunha,Leeds,0,7.333333,6.686341
2866,Haaland,Ipswich Town,1,7.000000,6.589538
2347,Bogle,Man Utd,1,7.000000,6.589538
870,Kostoulas,Crystal Palace,1,7.000000,6.589538
1857,Affengruber,Hull City,1,5.000000,5.595159
666,Yarmoliuk,Liverpool,1,5.000000,5.595159
2461,Trafford,Man Utd,1,5.000000,5.595159


In [20]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Grade V3 using the hidden test data
predictions_v3 = rf_model_v3.predict(X_test_v3)

mae_v3 = mean_absolute_error(y_test_v3, predictions_v3)
rmse_v3 = np.sqrt(mean_squared_error(y_test_v3, predictions_v3))

print(f"V3 Mean Absolute Error (MAE): {mae_v3:.2f} points")
print(f"V3 Root Mean Squared Error (RMSE): {rmse_v3:.2f} points")

V3 Mean Absolute Error (MAE): 2.08 points
V3 Root Mean Squared Error (RMSE): 2.91 points


In [23]:
# 1. Map the player prices from our original FPL dataset
price_map = dict(zip(df['id'], df['now_cost'] / 10))
future_games_df['price'] = future_games_df['element'].map(price_map)

# 2. Calculate ROI (Return on Investment)
# How many points are you projected to get per million spent?
future_games_df['value_per_million'] = future_games_df['projected_points'] / future_games_df['price']

# 3. Build a Budget Filter
# Simulate a user searching for the best value players who cost £7.5m or less
affordable_targets = future_games_df[future_games_df['price'] <= 7.5]

# 4. Sort by the best value (ROI) rather than just raw points
top_value_targets = affordable_targets.sort_values(by='value_per_million', ascending=False)

# THE FIX: We swapped 'adjusted_points' out for 'projected_points' here
columns_to_show = ['player_name', 'opponent_name', 'price', 'projected_points', 'value_per_million']

top_value_targets[columns_to_show].head(10)

,player_name,opponent_name,price,projected_points,value_per_million
744,De Cuyper,Crystal Palace,5.1,7.663242,1.502597
2347,Bogle,Man Utd,4.7,6.589538,1.402029
2146,O'Shea,Man City,4.0,5.386507,1.346627
798,Groß,Crystal Palace,6.0,7.663242,1.277207
1857,Affengruber,Hull City,4.5,5.595159,1.243369
2353,Gudmundsson,Man Utd,4.5,5.591197,1.242488
2359,Justin,Man Utd,4.5,5.591197,1.242488
251,McGinn,Newcastle,5.4,6.686341,1.238211
1145,Rushworth,Spurs,4.5,5.386507,1.197002
666,Yarmoliuk,Liverpool,4.8,5.595159,1.165658
